# Leak-free baseline models

Rebuilds the ticker-day panel with leakage fixes (leave-one-out market return for `beat_market`,
per-ticker expanding thresholds, lagged sentiment rolls). Then runs **Part 2 Step 5** only:
time-series CV comparing LogReg, LGBM, XGBoost, RF, and sklearn GradBoost — no aggressive
hyperparameter search yet.

**Next:** [`03-model-selection-advanced.ipynb`](03-model-selection-advanced.ipynb) — tuned XGB/LGBM,
ensembles, walk-forward, FinBERT + macro, Optuna.


In [1]:
%pip install -q xgboost
# imports for part 2

import pandas as pd                          # dataframes
import numpy as np                           # arrays and math
import copy                                  # deep-copy models so each CV fold starts fresh
import warnings                              # suppress noisy warnings
from pathlib import Path

from msa.utils.paths import (
    get_joined_dataset,
    get_model_selection_outputs_path,
    get_prices_daily_accumulated,
)
from scipy.stats import spearmanr            # rank correlation - better than pearson for noisy finance data

from sklearn.model_selection import (
    TimeSeriesSplit,                          # respects time order, never trains on future
    RandomizedSearchCV,                       # random hyperparameter search, way faster than grid
)
from sklearn.metrics import (
    accuracy_score,                           # fraction of correct predictions
    roc_auc_score,                            # area under ROC - measures ranking quality
    f1_score,                                 # harmonic mean of precision and recall
    mean_squared_error,                       # for RMSE check
)
from sklearn.linear_model import LogisticRegression  # baseline
from sklearn.ensemble import (
    RandomForestClassifier,                   # bagging ensemble
    GradientBoostingClassifier,               # sklearn's built-in boosting
)
from sklearn.preprocessing import StandardScaler     # z-score features for logistic regression
from sklearn.pipeline import Pipeline                # chain scaler + model together

from lightgbm import LGBMClassifier          # leaf-wise gradient boosting (fast)
from xgboost import XGBClassifier            # level-wise boosting (strong regularization)

warnings.filterwarnings("ignore")

print("imports ok")


ERROR: Operation cancelled by user
^C
Exception ignored in: <function _TemporaryFileCloser.__del__ at 0x7fa5892cd800>
Traceback (most recent call last):
  File "/home/trevor/.conda/envs/advds/lib/python3.11/tempfile.py", line 467, in __del__
    self.close()
  File "/home/trevor/.conda/envs/advds/lib/python3.11/tempfile.py", line 463, in close
    unlink(self.name)
KeyboardInterrupt: 
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'xgboost'

### step 1 — load data

same as part 1, loading the joined gdelt + ohlcv parquet. ret = intraday return (open to close).

In [4]:
# load the joined dataset (news articles matched to next-day prices)
df = pd.read_csv(get_joined_dataset().with_suffix(".csv"))
# df = pd.read_parquet(get_joined_dataset("parquet"))
df["article_date"] = pd.to_datetime(df["article_date"])  # ensure datetime type
df["price_date"]   = pd.to_datetime(df["price_date"])     # the trading day the article maps to

MAG7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]  # our universe
df = df[df["ticker"].isin(MAG7)].copy()                   # filter to MAG7 only
df = df[                                                   # 2-year window
    (df["article_date"] >= "2024-02-23") &
    (df["article_date"] <= "2026-02-23")
].copy()
df["ret"] = (df["next_close"] - df["next_open"]) / df["next_open"]  # intraday return

# Aggregate from article-level → ticker-day level
# Each row = 1 ticker on 1 trading day, with average sentiment and return
ticker_daily = (
    df.groupby(["ticker", "article_date", "price_date"])
    .agg(
        mean_sent  = ("sentiment_score", "mean"),     # average sentiment across all articles that day
        n_articles = ("sentiment_score", "count"),     # how many articles (coverage intensity)
        neg_ratio  = ("sentiment_label", lambda x: (x == "negative").mean()),  # fraction of negative articles
        pos_ratio  = ("sentiment_label", lambda x: (x == "positive").mean()),  # fraction of positive articles
        ret        = ("ret", "first"),                 # intraday return (same for all articles on same day)
        volume     = ("next_volume", "first"),         # trading volume
    )
    .reset_index()
    .sort_values("article_date")
    .drop_duplicates(subset=["ticker", "price_date"], keep="last")  # keep latest if dupes
    .sort_values(["ticker", "price_date"])             # sort by ticker then date for groupby operations
    .reset_index(drop=True)
)

print(f"Articles loaded: {len(df):,}")
print(f"Ticker-daily rows: {len(ticker_daily):,}")
print(f"Tickers: {sorted(ticker_daily['ticker'].unique())}")
print(f"Date range: {ticker_daily['price_date'].min()} -> {ticker_daily['price_date'].max()}")


Articles loaded: 89,019
Ticker-daily rows: 1,386
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
Date range: 2024-02-26 00:00:00 -> 2026-02-24 00:00:00


### step 2 — build price features (no look-ahead)

every feature uses only info available BEFORE the trading day. we shift(1) everything so the model never sees todays data when predicting todays return. all rolling windows start from the shifted series.

In [6]:
# build price-based features from the clean daily price panel
prices = pd.read_csv(get_prices_daily_accumulated().with_suffix(".csv"))
# prices = pd.read_parquet(get_prices_daily_accumulated("parquet"))
prices["date"] = pd.to_datetime(prices["date"])           # ensure datetime
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)  # sort for correct shifting

# Close-to-close return: how much did the stock move from yesterday's close to today's close?
prices["ret_close"] = prices.groupby("ticker")["close"].pct_change()

# Create a groupby object — all .shift() and .rolling() are PER TICKER
g = prices.groupby("ticker")

# ── MOMENTUM FEATURES ──
# shift(1) = yesterday's value. This ensures NO look-ahead bias.
prices["mom_1d"]     = g["ret_close"].shift(1)            # yesterday's close-to-close return
prices["mom_2d"]     = g["ret_close"].shift(2)            # return from 2 days ago
prices["mom_5d_avg"] = g["ret_close"].transform(           # average of last 5 days' returns
    lambda x: x.shift(1).rolling(5).mean()                 # shift first, THEN roll → no leakage
)

# ── VOLATILITY FEATURES ──
prices["vol_10d"] = g["ret_close"].transform(              # 10-day rolling standard deviation of returns
    lambda x: x.shift(1).rolling(10).std()                 # measures recent price turbulence
)
prices["vol_20d"] = g["ret_close"].transform(              # 20-day rolling std (longer-term vol)
    lambda x: x.shift(1).rolling(20).std()
)

# ── VOLUME Z-SCORE ──
# How unusual is today's volume compared to the recent 10-day average?
# z > 2 means volume is 2 standard deviations above normal → something big happening
prices["volume_z"] = (
    (g["volume"].shift(1) - g["volume"].transform(lambda x: x.shift(1).rolling(10).mean()))
    / g["volume"].transform(lambda x: x.shift(1).rolling(10).std())
)

# ── RSI (Relative Strength Index) ──
# Classic momentum oscillator: 0-100 scale. >70 = overbought, <30 = oversold.
def compute_rsi(series, period=14):
    delta = series.diff()                                  # daily price change
    gain = delta.clip(lower=0).rolling(period).mean()      # average gain over `period` days
    loss = (-delta.clip(upper=0)).rolling(period).mean()   # average loss over `period` days
    rs = gain / loss                                       # relative strength ratio
    return 100 - (100 / (1 + rs))                          # transform to 0-100 scale

prices["rsi_14"] = (
    g["close"].transform(lambda x: compute_rsi(x, 14))    # compute RSI per ticker
    .groupby(prices["ticker"]).shift(1)                    # shift by 1 to avoid look-ahead
)

# ── DISTANCE FROM MOVING AVERAGE ──
# How far is the stock from its 10-day moving average? (mean reversion signal)
prices["ma_10"] = g["close"].transform(
    lambda x: x.rolling(10).mean()                         # 10-day simple moving average
).groupby(prices["ticker"]).shift(1)                       # lag it
prices["close_prev"] = g["close"].shift(1)                 # yesterday's close
prices["dist_from_ma10"] = (                               # % distance from MA
    (prices["close_prev"] - prices["ma_10"]) / prices["ma_10"]
)

print(f"Price panel: {len(prices):,} rows, {prices['ticker'].nunique()} tickers")
print(f"Date range: {prices['date'].min()} -> {prices['date'].max()}")


Price panel: 3,584 rows, 7 tickers
Date range: 2024-02-08 00:00:00 -> 2026-02-24 00:00:00


### step 3 — merge + build sentiment features (fixing the leakage)

this is where we fix the 3 leakage issues from part 1:
1. `sent_5d_avg` — now properly lagged (shift first then roll)
2. `beat_market` target — fixed so it doesnt include the tickers own return in the market avg
3. expanding stats — now computed per ticker instead of mixing all tickers together on same date

In [7]:
# merge price features + build sentiment features (leak-free version)
price_features = prices[[
    "date", "ticker",
    "mom_1d", "mom_2d", "mom_5d_avg",                     # momentum
    "vol_10d", "vol_20d", "volume_z",                      # volatility & volume
    "rsi_14", "dist_from_ma10",                            # technical indicators
]].rename(columns={"date": "price_date"})                  # rename for merge key

# LEFT JOIN: attach price features to each ticker-day sentiment row
td = ticker_daily.merge(price_features, on=["price_date", "ticker"], how="left")

# ── SENTIMENT FEATURES ──
# 5-day rolling average of sentiment, LAGGED by 1 day
# shift(1) ensures we don't use today's sentiment to predict today's return
td["sent_5d_avg"]   = td.groupby("ticker")["mean_sent"].transform(
    lambda x: x.shift(1).rolling(5).mean()
)
# Sentiment surprise = today's sentiment minus the recent average
# This measures "was today unusually positive/negative vs the trend?"
td["sent_surprise"] = td["mean_sent"] - td["sent_5d_avg"]

# 3-day sentiment momentum = how sentiment has been changing
td["sent_mom_3d"]   = td.groupby("ticker")["mean_sent"].transform(
    lambda x: x.diff(3)                                    # current - value from 3 days ago
)

# ── EXPANDING NEG_EXTREME (per-ticker to avoid cross-contamination) ──
# Mark days where sentiment is in the bottom 20% of what we've seen SO FAR for THIS ticker
# Using expanding (not rolling) means the threshold adapts as we see more data
td = td.sort_values(["ticker", "price_date"]).reset_index(drop=True)
td["neg_extreme_20"] = td.groupby("ticker")["mean_sent"].transform(
    lambda x: (x <= x.expanding().quantile(0.20)).astype(float)
)

# ── INTERACTION FEATURES ──
# These combine sentiment with price momentum to capture conditional effects
td["neg20_x_mom1d"]     = td["neg_extreme_20"] * td["mom_1d"]    # extreme negative news + momentum
td["sent_x_mom1d"]      = td["mean_sent"] * td["mom_1d"]         # raw sentiment × momentum
td["neg_ratio_x_mom1d"] = td["neg_ratio"] * td["mom_1d"]         # negative article % × momentum

# ── FIX: LEAVE-ONE-OUT MARKET RETURN ──
# Old (leaky): mkt_ret = mean of all 7 tickers including the ticker being predicted
# New (clean): for each ticker, compute market return excluding that ticker
date_sums = td.groupby("price_date").agg(
    total_ret = ("ret", "sum"),                            # sum of all 7 tickers' returns
    n_tickers = ("ret", "count"),                          # how many tickers on that date
).reset_index()
td = td.merge(date_sums, on="price_date", how="left")
# For each ticker: mkt_ret = (sum of all returns - this ticker's return) / (n - 1)
td["mkt_ret_loo"] = (td["total_ret"] - td["ret"]) / (td["n_tickers"] - 1)
td["relative_ret"] = td["ret"] - td["mkt_ret_loo"]        # did this ticker outperform peers?
td["beat_market"]  = (td["relative_ret"] > 0).astype(int)  # binary target: 1 = outperformed
td.drop(columns=["total_ret", "n_tickers"], inplace=True)  # clean up temp columns

# ── DIRECTION TARGET ──
td["ticker_code"] = td["ticker"].astype("category").cat.codes  # numeric encoding for ticker
td["up"] = (td["ret"] > 0).astype(int)                    # 1 = positive return, 0 = negative

print(f"Final table: {len(td):,} rows")
print(f"Beat-market balance: {td['beat_market'].mean():.3f}")  # should be near 0.50
print(f"Direction balance: {td['up'].mean():.3f}")

Final table: 1,386 rows
Beat-market balance: 0.419
Direction balance: 0.511


### step 4 — define features and clean data

two feature sets:
- `top_features` = the 7 strongest signals from our correlation scan + ticker code. less overfitting risk
- `full_features` = all 19 features. richer but noisier on ~1300 rows

we let the tree models figure out which features actually matter instead of hardcoding weights like before.

In [ ]:
# define our two feature sets and clean the data
full_features = [
    "rsi_14", "mom_1d", "mom_2d", "mom_5d_avg",           # price momentum features
    "volume_z", "vol_10d", "vol_20d",                      # volume and volatility
    "dist_from_ma10",                                      # mean reversion
    "mean_sent", "sent_5d_avg", "sent_surprise",           # sentiment level features
    "sent_mom_3d",                                         # sentiment trend
    "neg_ratio", "pos_ratio", "n_articles",                # article coverage
    "neg_extreme_20",                                      # sentiment regime flag
    "sent_x_mom1d", "neg_ratio_x_mom1d", "neg20_x_mom1d", # interaction features
    "ticker_code",                                         # which stock (categorical)
]

# TOP features: only the statistically significant signals
top_features = [
    "rsi_14", "mom_1d", "volume_z",                        # strongest individual signals
    "sent_mom_3d", "dist_from_ma10",                       # near-significant signals
    "sent_x_mom1d",                                        # best interaction term
    "ticker_code",                                         # stock identity
]

# Drop rows with any NaN in our features (mostly from rolling window warm-up)
clean = td[full_features + ["ret", "up", "beat_market", "price_date", "ticker"]].dropna()
clean = clean.sort_values("price_date").reset_index(drop=True)  # sort by time for TimeSeriesSplit

print(f"Clean rows: {len(clean):,} (dropped {len(td) - len(clean):,} NaN)")
print(f"Direction balance: {clean['up'].mean():.3f}")     # ~0.50 = balanced classes
print(f"Beat-market balance: {clean['beat_market'].mean():.3f}")

### step 5 — model comparison (LightGBM vs XGBoost vs baselines)

using TimeSeriesSplit(n_splits=5, gap=5) because financial data is ordered in time. if we did random k-fold the model would train on future and test on past = leakage. the 5-day gap prevents spillover from correlated adjacent days.

we test everything: logistic regression as baseline, two lightgbm configs, two xgboost configs, random forest, and gradient boosting. the goal is to see which architecture gets closest to 65% AUC.

In [ ]:
# model comparison — testing everything against each other

tscv = TimeSeriesSplit(n_splits=5, gap=5)  # 5 expanding folds with 5-day gap
models = {
    # logistic regression: simple linear baseline
    "LogReg_L2 (top)": ("top", Pipeline([
        ("scaler", StandardScaler()),          # LR needs scaled features (tree models don't)
        ("clf", LogisticRegression(
            C=0.1,                             # regularization strength (lower = more regularized)
            max_iter=1000,                     # enough iterations to converge
            random_state=42,                   # reproducibility
        )),
    ])),

    "LogReg_L2 (full)": ("full", Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=0.1, max_iter=1000, random_state=42)),
    ])),

    # lightgbm conservative = heavy regularization to avoid overfitting
    "LGBM_conservative (full)": ("full", LGBMClassifier(
        n_estimators=300,                      # number of boosting rounds (trees)
        max_depth=3,                           # shallow trees → less overfitting
        learning_rate=0.02,                    # small steps → more stable but needs more trees
        num_leaves=8,                          # max leaves per tree (2^3=8 matches depth=3)
        min_child_samples=30,                  # each leaf needs ≥30 samples → no tiny leaves
        subsample=0.7,                         # use 70% of rows per tree (row subsampling)
        colsample_bytree=0.7,                  # use 70% of features per tree (feature subsampling)
        reg_alpha=2.0,                         # L1 regularization → pushes weak features to zero
        reg_lambda=2.0,                        # L2 regularization → shrinks all weights
        random_state=42,
        verbose=-1,                            # suppress training logs
    )),

    # lightgbm moderate = slightly more complexity
    "LGBM_moderate (full)": ("full", LGBMClassifier(
        n_estimators=500,                      # more trees (compensated by smaller learning rate)
        max_depth=4,                           # slightly deeper trees
        learning_rate=0.01,                    # even smaller steps
        num_leaves=15,                         # more leaves allows more complex patterns
        min_child_samples=30,
        subsample=0.8,                         # slightly less aggressive subsampling
        colsample_bytree=0.7,
        reg_alpha=1.0,                         # lighter regularization
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )),

    # xgboost conservative = level-wise boosting with strong regularization
    "XGB_conservative (full)": ("full", XGBClassifier(
        n_estimators=300,                      # number of boosting rounds
        max_depth=3,                           # shallow trees (XGB grows level-wise by default)
        learning_rate=0.02,                    # small step size
        subsample=0.7,                         # row subsampling
        colsample_bytree=0.7,                  # feature subsampling per tree
        min_child_weight=30,                   # similar to min_child_samples in LGBM
        gamma=1.0,                             # min loss reduction to make a split (XGB-specific)
        reg_alpha=2.0,                         # L1 regularization
        reg_lambda=2.0,                        # L2 regularization
        eval_metric="logloss",                 # metric for training (binary cross-entropy)
        random_state=42,
        verbosity=0,                           # suppress output
    )),

    # xgboost moderate = more capacity
    "XGB_moderate (full)": ("full", XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=30,
        gamma=0.5,                             # less penalty for splits → allows more complexity
        reg_alpha=1.0,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=42,
        verbosity=0,
    )),

    # xgboost with top features only (less overfitting risk)
    "XGB_conservative (top)": ("top", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.02,
        subsample=0.7,
        colsample_bytree=0.8,                  # higher since fewer features
        min_child_weight=30,
        gamma=1.0,
        reg_alpha=2.0,
        reg_lambda=2.0,
        eval_metric="logloss",
        random_state=42,
        verbosity=0,
    )),

    # random forest baseline
    "RandomForest (full)": ("full", RandomForestClassifier(
        n_estimators=300,                      # number of independent trees
        max_depth=4,                           # shallow trees
        min_samples_leaf=30,                   # each leaf needs ≥30 samples
        max_features="sqrt",                   # each tree sees sqrt(n_features) features
        random_state=42,
    )),

    # gradient boosting (sklearn built-in)
    "GradBoost (full)": ("full", GradientBoostingClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.02,
        min_samples_leaf=30,
        subsample=0.8,
        random_state=42,
    )),
}

# cross-validation loop
results_v2 = []

for target_name, target_col in [("direction", "up"), ("beat_market", "beat_market")]:
    print(f"\n{'*' * 70}")
    print(f"TARGET: {target_name}")
    print(f"{'*' * 70}")

    for model_name, (feat_key, model_template) in models.items():
        # Select feature set based on model config
        feats = top_features if feat_key == "top" else full_features

        accs, aucs, f1s = [], [], []                       # collectors for each fold's metrics

        for tr_idx, te_idx in tscv.split(clean):           # iterate over time-series folds
            X_tr = clean.iloc[tr_idx][feats].values        # training features (numpy array)
            X_te = clean.iloc[te_idx][feats].values        # test features
            y_tr = clean.iloc[tr_idx][target_col].values   # training labels
            y_te = clean.iloc[te_idx][target_col].values   # test labels

            m = copy.deepcopy(model_template)              # fresh copy of the model
            m.fit(X_tr, y_tr)                              # train on this fold's training set

            pred  = m.predict(X_te)                        # hard predictions (0 or 1)
            proba = m.predict_proba(X_te)[:, 1]            # probability of class 1

            accs.append(accuracy_score(y_te, pred))        # fraction correct
            aucs.append(roc_auc_score(y_te, proba))        # ranking quality
            f1s.append(f1_score(y_te, pred))               # precision/recall balance

        # Store average across all folds
        row = {
            "target": target_name, "model": model_name,
            "acc": np.mean(accs), "acc_std": np.std(accs),
            "auc": np.mean(aucs), "auc_std": np.std(aucs),
            "f1": np.mean(f1s),
        }
        results_v2.append(row)
        print(f"  {model_name:<30} acc={row['acc']:.4f}({row['acc_std']:.3f})  "
              f"auc={row['auc']:.4f}({row['auc_std']:.3f})  f1={row['f1']:.4f}")

results_v2_df = pd.DataFrame(results_v2)
print("\n── Best models by AUC ──")
print(results_v2_df.sort_values("auc", ascending=False).head(10).to_string(index=False))

In [ ]:
OUT_DIR = get_model_selection_outputs_path()
OUT_DIR.mkdir(parents=True, exist_ok=True)
results_v2_df.to_pickle(OUT_DIR / 'part2_baseline_results_df.pkl')
print('Wrote', OUT_DIR / 'part2_baseline_results_df.pkl')
